In [1]:
import numpy as np
import torch
import torch.nn.functional as F
import torch.nn as nn
import torchaudio
import sys
import math

In [2]:
from torch.utils.data import DataLoader, Dataset

class CustomAudioDataset(Dataset):
    def __init__(self, file_list, transcript_list):
        
        self.file_list = file_list
        self.transcript_list = transcript_list

    def __len__(self):
        return len(self.file_list)
    
    def __getitem__(self, idx):
        audio_path = self.file_list[idx]
        transcript = self.transcript_list[idx]
        waveform, sample_rate = torchaudio.load(audio_path)
        return waveform, sample_rate, transcript

In [3]:
def flip(x, dim):
    xsize = x.size()
    dim = x.dim() + dim if dim < 0 else dim
    x = x.contiguous()
    x = x.view(-1, *xsize[dim:])
    x = x.view(x.size(0), x.size(1), -1)[:, getattr(torch.arange(x.size(1)-1, -1, -1), ('cpu','cuda')[x.is_cuda])().long(), :]
    return x.view(xsize)

def sinc(band,t_right):
    y_right= torch.sin(2*math.pi*band*t_right)/(2*math.pi*band*t_right)
    y_left= flip(y_right,0)

    y=torch.cat([y_left,torch.ones(1),y_right]) # makes the mathematical condition here

    return y

In [4]:
class SincConv_fast(nn.Module):
    @staticmethod
    def to_mel(hz):
        hz = torch.tensor(hz, dtype=torch.float32)
        return 2595 * torch.log10(1 + hz / 700)

    @staticmethod
    def to_hz(mel):
        mel = torch.tensor(mel, dtype=torch.float32)
        return 700 * (10 ** (mel / 2595) - 1)

    def __init__(self, out_channels, kernel_size, sample_rate=16000, in_channels=1, stride=1, padding=0, dilation=1, bias=False, groups=1, min_low_hz=50, min_band_hz=50):

        super(SincConv_fast, self).__init__()
        if in_channels !=1:
            msg = "SincConv only support one input channel (here, in_channels = {%i})"%(in_channels)
            raise ValueError(msg)

        self.out_channels = out_channels
        self.kernel_size = kernel_size

        if kernel_size%2==0:
            self.kernel_size = kernel_size+1

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

        if bias:
            raise ValueError("SincConv does not bias")
        if groups >1:
            raise ValueError('SincConv does not suppooer groups')

        self.sample_rate = sample_rate
        self.min_low_hz = min_low_hz
        self.min_band_hz = min_band_hz

        low_hz = 30
        high_hz = self.sample_rate / 2 - (self.min_low_hz + self.min_band_hz)

        mel = np.linspace(self.to_mel(low_hz),self.to_mel(high_hz),self.out_channels+1)
        hz = self.to_hz(mel)

        # filter lower frequency (out_channels, 1)
        self.low_hz_ = nn.Parameter(torch.Tensor(hz[:-1]).view(-1,1))

        # filter frequency band (out_channels, 1)
        self.band_hz_ = nn.Parameter(torch.Tensor(np.diff(hz)).view(-1,1))

        # Hamming window
        n_lin = torch.linspace(0, (self.kernel_size/2)-1, steps=int((self.kernel_size/2))) # computing only half of the window
        window_ = 0.54-0.46*torch.cos(2*math.pi*n_lin/self.kernel_size)

        # (1, kernel_size/2)
        n = (self.kernel_size - 1) / 2.0
        n_ = 2*math.pi*torch.arange(-n,0).view(1,-1) / self.sample_rate # due to symmerty, I only need half of the time axes

        self.register_buffer("window_", window_)
        self.register_buffer("n_", n_)

    def forward(self, waveforms):
        # device = waveforms.device
        # self.n_ = self.n_.to(device)
        # self.window_ = self.window_.to(device)

        low = self.min_low_hz + torch.abs(self.low_hz_)

        high = torch.clamp(low + self.min_band_hz + torch.abs(self.band_hz_),self.min_low_hz,self.sample_rate/2)
        band=(high-low)[:,0]

        f_times_t_low = torch.matmul(low,self.n_)
        f_times_t_high = torch.matmul(high,self.n_)

        band_pass_left = ((torch.sin(f_times_t_high)-torch.sin(f_times_t_low))/(self.n_/2))*self.window_
        band_pass_center = 2*band.view(-1,1)
        band_pass_right = torch.flip(band_pass_left,dims=[1])

        band_pass = torch.cat([band_pass_left, band_pass_center, band_pass_right], dim=1)

        band_pass = band_pass / (2*band[:,None])

        self.filters = (band_pass).view(self.out_channels, 1, self.kernel_size)

        return F.conv1d(waveforms, self.filters, stride=self.stride, padding=self.padding, dilation=self.dilation, bias=None, groups=1)


In [5]:
class SinusoidalPositionEncoding(nn.Module):
    def __init__(self, embed_size):
        super().__init__()
        self.embed_size = embed_size

    def forward(self, x):
        # x: [B, T, C]
        B, T, C = x.size()

        # generate PE dynamically
        position = torch.arange(T, device=x.device).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, C, 2, device=x.device) * (-math.log(10000.0) / C))

        pe = torch.zeros(T, C, device=x.device)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        return x + pe  # [B, T, C]

In [6]:
from turtle import forward


class Head(nn.Module):
    """One head of self-attention"""
    def __init__(self, n_embd, head_size):
        super().__init__()

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(0.2)

    def forward(self, embedding):
        B,T,C = embedding.shape
        k = self.key(embedding)
        # print(k.shape)
        # print(k)
        q = self.query(embedding)
        # print(q.shape)
        # print(q)
        scale = q.size(-1)**0.5
        wei = (q @ k.transpose(-2,-1))/scale
        wei = F.softmax(wei,dim=-1)
        wei = self.dropout(wei)
        v = self.value(embedding)
        out = wei @ v

        return out

class MultiHeadAttention(nn.Module):
    """multiple heads of self-attention in parallel"""

    def __init__(self, num_heads, head_size, n_embd):
        super().__init__()
        self.heads = nn.ModuleList([Head(n_embd,head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self,x):
        # print(x.shape)
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out
    
class FeedForward(nn.Module):
    """a simple linear layer"""
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(0.2)
        )

    def forward(self,x):
        return self.net(x)
    
class Encoder(nn.Module):
    """Transformer Encoder Block"""
    def __init__(self, n_embd, n_heads):
        super().__init__()
        head_size = n_embd // n_heads
        self.sa = MultiHeadAttention(n_heads, head_size, n_embd)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self,x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [7]:
class DecoderHead(nn.Module):
    """one head of self-attention"""
    def __init__(self, n_embd, head_size, block_size=1024):
        super().__init__()

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(0.2)
        
    def forward(self, embedding):
        B,T,C = embedding.shape
        k = self.key(embedding)
        # print(k.shape)
        # print(k)
        q = self.query(embedding)
        # print(q.shape)
        # print(q)
        scale = q.size(-1)**0.5
        wei = (q @ k.transpose(-2,-1))/scale
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(embedding)
        out = wei @ v

        return out
    
class MultiHeadAttentionDecoder(nn.Module):
    """multiple heads of self-attention in parallel"""

    def __init__(self, num_heads, head_size, n_embd):
        super().__init__()
        self.heads = nn.ModuleList([DecoderHead(n_embd,head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self,x):
        # print(x.shape)
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out
    
class FeedForwardDecoder(nn.Module):
    """a simple linear layer"""
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(0.2)
        )

    def forward(self,x):
        return self.net(x)
    
class CrossAttentionHead(nn.Module):
    """One head of cross-attention"""
    def __init__(self, n_embd, head_size):
        super().__init__()

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(0.2)
        
    def forward(self, embedding_q, embedding_kv):
        B,T,C = embedding_q.shape
        k = self.key(embedding_kv)
        # print(k.shape)
        # print(k)
        q = self.query(embedding_q)
        # print(q.shape)
        # print(q)
        scale = q.size(-1)**0.5
        wei = (q @ k.transpose(-2,-1))/scale
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(embedding_kv)
        out = wei @ v

        return out
    
class MultiHeadCrossAttention(nn.Module):
    """multiple head of cross attention in parallel"""
    def __init__(self, num_heads, head_size, n_embd):
        super().__init__()
        self.heads = nn.ModuleList([CrossAttentionHead(n_embd,head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self, x, y):
        out = torch.cat([h(x,y) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out
    
class Decoder(nn.Module):
    """Transformer Decoder Block"""
    def __init__(self, n_embd, n_heads):
        super().__init__()
        head_size = n_embd // n_heads
        self.self_sa = MultiHeadAttentionDecoder(n_heads, head_size, n_embd)
        self.cross_sa = MultiHeadCrossAttention(n_heads, head_size, n_embd)
        self.ffwd = FeedForwardDecoder(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ln3 = nn.LayerNorm(n_embd)

        # self.tril = torch.tril(torch.ones(1000, 1000))

    def forward(self,x, enco_out):
        x = x + self.self_sa(self.ln1(x))
        x = x + self.cross_sa(self.ln2(x), enco_out)
        x = x + self.ffwd(self.ln3(x))
        return x

In [ ]:
class SincNet3Layer(nn.Module):
    def __init__(self, sample_rate, out_channels=384):
        super(SincNet3Layer, self).__init__()

        # dec_out_channel = 384

        self.act1  = nn.GELU()
        self.sinc1 = SincConv_fast(out_channels=4*out_channels, kernel_size=251, sample_rate=sample_rate, stride=1, padding=0)
        # self.sinc2 = SincConv_fast(out_channels=2*out_channels, kernel_size=100, sample_rate=48000, stride=2, padding=0)
        self.pool = nn.MaxPool1d(kernel_size=3, stride=3)
        self.conv1 = nn.Conv1d(in_channels=4*out_channels, out_channels=2*out_channels, kernel_size=5, stride=2)
        self.conv2 = nn.Conv1d(in_channels=2*out_channels, out_channels=out_channels, kernel_size=10, stride=3)
        self.token_embedding = nn.Embedding(vocab_size, out_channels)
        self.pos1 = SinusoidalPositionEncoding(embed_size=out_channels)
        self.pos2 = SinusoidalPositionEncoding(embed_size=out_channels)
        num_heads = 12
        num_blocks = 4
        self.encoder = nn.Sequential(*[Encoder(n_embd=out_channels, n_heads=num_heads) for _ in range(num_blocks)])
        self.decoder = nn.ModuleList([Decoder(n_embd=out_channels, n_heads=num_heads) for _ in range(num_blocks)])
        self.lm_head = nn.Linear(out_channels, vocab_size)

    def forward(self,x,y,targets=None):
        out = self.sinc1(x)  # Shape: [batch, channels, time]
        out = self.pool(out)  # pooling
        out = self.conv1(out)  # Sinc convolution2
        out = self.pool(out)  # pooling
        out = self.conv2(out)  # convolution
        out = self.act1(out)
        out = self.pool(out)
        out = out.transpose(1, 2)  # Shape: [batch, time, channels]
        out = self.pos1(out)
        enc_out = self.encoder(out)

        tok_emd = self.token_embedding(y)
        tok_pos = self.pos2(tok_emd)

        dec = tok_pos
        for layer in self.decoder:
           dec = layer(dec, enc_out)
        logits = self.lm_head(dec)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.reshape(B*T,C)
            # print(logits)
            targets = targets.reshape(B*T)
            # print(targets)
            
            loss = F.cross_entropy(logits, targets)

        return logits, loss

In [ ]:
import torch.optim as optim

learning_rate = 1e-4

model = SincNet3Layer(sample_rate=48000)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)